In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import time
from sklearn.preprocessing import normalize
import google.generativeai as genai
from dotenv import load_dotenv

# ==============================================================================
# [1] 환경 설정
# ==============================================================================
load_dotenv() # .env 파일 로드

# API 키 가져오기
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 키가 없는 경우 에러 발생
if not GOOGLE_API_KEY:
    raise ValueError("❌ API Key를 찾을 수 없습니다. .env 파일을 확인하거나 키를 직접 입력하세요.")

# Gemini 설정
genai.configure(api_key=GOOGLE_API_KEY)
EMBEDDING_MODEL = "models/text-embedding-004"

# ==============================================================================
# [2] 데이터셋 (TONE_DICT)
# ==============================================================================
TONE_DICT = {
    'Scientific': ['연구', '데이터', '기술', '특허', '임상', '메커니즘', '효능', '성분', '솔루션', '혁신', '분석', '검증'],
    'Emotional': ['사랑', '행복', '마음', '위로', '함께', '추억', '소중한', '느낌', '감동', '선물', '일상', '여유'],
    'Luxury': ['프리미엄', '고품격', '가치', '특별한', '최고', '럭셔리', '노블레스', '장인', '헤리티지', '압도적'],
    'Casual': ['진짜', '대박', '완전', '가성비', '꿀팁', '그냥', '솔직', '친구', '쉬운', '간편', '추천']
}

# ==============================================================================
# [3] [핵심 수정] 단일 텍스트 임베딩 함수
# ==============================================================================
def get_embedding_gemini(text):
    """
    단어 하나를 받아 벡터 하나를 반환합니다.
    """
    try:
        # task_type="clustering": 유사도 비교나 군집화에 적합한 벡터 생성
        result = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=text,
            task_type="clustering" 
        )
        return result['embedding']
    except Exception as e:
        print(f"⚠️ '{text}' 임베딩 실패: {e}")
        return None

# ==============================================================================
# [4] [핵심 수정] 벡터 생성 로직 (단어별 순회 방식 적용)
# ==============================================================================
def build_tone_assets(tone_dict):
    tone_vectors = {}
    meta_rows = []

    print(f"🚀 Start processing {len(tone_dict)} tones using Gemini API...")

    for tone_label, keywords in tone_dict.items():
        print(f" -> Processing '{tone_label}'...", end="")
        
        # 1. 단어 하나씩 루프를 돌며 벡터를 수집합니다.
        vectors_list = []
        for word in keywords:
            vec = get_embedding_gemini(word)
            if vec is not None:
                vectors_list.append(vec)
            # (선택 사항) API 호출 속도 조절이 필요하면 주석 해제
            # time.sleep(0.1) 

        if not vectors_list:
            print(" ❌ 실패 (모든 키워드 임베딩 실패)")
            continue

        # 2. 수집된 벡터들을 하나의 배열로 변환 (Shape: 단어개수 x 768)
        embeddings = np.array(vectors_list)
        
        # 3. Centroid (평균 벡터) 계산
        # axis=0: 각 차원(열)끼리 평균을 냄
        centroid = np.mean(embeddings, axis=0)
        
        # 4. Normalization (L2 정규화)
        if np.linalg.norm(centroid) > 0:
            centroid_norm = normalize(centroid.reshape(1, -1), norm='l2')[0]
        else:
            centroid_norm = centroid

        # 5. 결과 저장
        tone_vectors[tone_label] = centroid_norm
        
        meta_rows.append({
            "tone_id": tone_label,
            "keyword_count": len(vectors_list),
            "keywords": ", ".join(keywords),
            "model_used": EMBEDDING_MODEL
        })
        
        print(f" ✅ Complete. (Shape: {centroid_norm.shape})")

    return tone_vectors, pd.DataFrame(meta_rows)

# ==============================================================================
# [5] 실행 및 검증
# ==============================================================================
if __name__ == "__main__":
    try:
        vectors_pkl, df_meta = build_tone_assets(TONE_DICT)
        
        # 저장
        with open("tone_vectors.pkl", "wb") as f:
            pickle.dump(vectors_pkl, f)
        print("\n📂 [Saved] 'tone_vectors.pkl' saved successfully.")
        
        df_meta.to_csv("tone_metadata.csv", index=False, encoding="utf-8-sig")
        print("📂 [Saved] 'tone_metadata.csv' saved successfully.")
        
        # [검증] Scientific vs Luxury 벡터가 다른지 확인
        print("\n🔍 [데이터 검증]")
        vec_sci = vectors_pkl['Scientific']
        vec_lux = vectors_pkl['Luxury']
        
        print(f"Scientific (앞 5자리): {vec_sci[:5]}")
        print(f"Luxury     (앞 5자리): {vec_lux[:5]}")
        
        if np.allclose(vec_sci, vec_lux, atol=1e-6):
            print("\n🚨 [경고] 벡터 값이 여전히 똑같습니다! 로직을 다시 확인하세요.")
        else:
            print("\n✅ [성공] Scientific과 Luxury 벡터가 서로 다르게 잘 생성되었습니다.")

    except Exception as e:
        print(f"\n❌ 작업 중단됨: {e}")

🚀 Start processing 4 tones using Gemini API...
 -> ✅ 'Scientific' complete.
 -> ✅ 'Emotional' complete.
 -> ✅ 'Luxury' complete.
 -> ✅ 'Casual' complete.

📂 [Saved] 'tone_vectors.pkl' saved successfully.
📂 [Saved] 'tone_metadata.csv' saved successfully.
